In [ ]:
import pandas as pd
workers_and_managers = pd.DataFrame(
    data=[
        (1, 'Boss', 1),
        (3, 'Alice', 3),
        (2, 'Bob', 1),
        (4, 'Daniel', 2),
        (7, 'Luis', 4),
        (8, 'Jhon', 3),
        (9, 'Angela', 8),
        (77, 'Robert', 1)
    ],
    columns=['employee_id', 'employee_name', 'manager_id']
).astype({
    'employee_id': 'int64',
    'employee_name': 'string',
    'manager_id': 'int64'
})
workers_and_managers

,employee_id,employee_name,manager_id
0,1,Boss,1
1,3,Alice,3
2,2,Bob,1
3,4,Daniel,2
4,7,Luis,4
5,8,Jhon,3
6,9,Angela,8
7,77,Robert,1


Необходимо найти

employee_id тех работников, которые отчитываются боссу напрямую или через других менеджеров (например, Даниэль отчитывается Бобу, а Боб отчитывается Боссу, условие выполнено). Джон отчитывается Элис, а Элис никому не отчитывается, условие не выполнено. Между работником и боссом может быть максимум 2 промежуточных менеджера. Босса в результирующий запрос включать не нужно

Ожидаемые колонки в результирующем запросе:

employee_id,
employee_name

In [ ]:
df = workers_and_managers
level_1 = df.merge(df, left_on='manager_id', right_on='employee_id')
level_2 = level_1.merge(df, left_on='manager_id_y', right_on='employee_id')
result = level_2[(level_2['employee_name_x'] != 'Boss') & (level_2['manager_id'] == 1)]
result = result[['employee_id_x', 'employee_name_x']].rename(columns={'employee_id_x': 'employee_id', 'employee_name_x': 'employee_name'})
result

,employee_id,employee_name
2,2,Bob
3,4,Daniel
4,7,Luis
7,77,Robert


3.4=================================

In [ ]:
Employees_Russia = pd.DataFrame(
    data=[
        (1, 'Иван'),
        (2, 'Петр'),
        (3, 'Мария')
    ],
    columns=['id', 'name']
).astype({
    'id': 'int64',
    'name': 'string'
})

Employees_USA = pd.DataFrame(
    data=[
        (1, 'John'),
        (2, 'Peter'),
        (3, 'Mary')
    ],
    columns=['id', 'name']
).astype({
    'id': 'int64',
    'name': 'string'
})
Employees_Russia

,id,name
0,1,Иван
1,2,Петр
2,3,Мария


In [ ]:
Employees_USA

,id,name
0,1,John
1,2,Peter
2,3,Mary


Напишите SQL запрос для получения уникальных имен из обеих таблиц

Ожидаемый формат данных в результирующем запросе: name

In [ ]:
union_df = pd.concat([Employees_Russia, Employees_USA], ignore_index=True)
union_df

,id,name
0,1,Иван
1,2,Петр
2,3,Мария
3,1,John
4,2,Peter
5,3,Mary


4.2=========================================

In [ ]:
sales = pd.DataFrame({
    'sale_id': [1, 2, 3, 4, 5],
    'manager_id': [101, 101, 102, 102, 101],
    'sale_date': ['2023-01-01', '2023-01-02', '2023-01-01', '2023-01-03', '2023-01-03'],
    'amount': [100, 200, 300, 150, 250]
})

sales['sale_date'] = pd.to_datetime(sales['sale_date'])

sales = sales.astype({
    'sale_id': 'int64',
    'manager_id': 'int64',
    'amount': 'int64'
})
sales

,sale_id,manager_id,sale_date,amount
0,1,101,2023-01-01,100
1,2,101,2023-01-02,200
2,3,102,2023-01-01,300
3,4,102,2023-01-03,150
4,5,101,2023-01-03,250


Задание:
Напишите SQL-запрос с использованием оконных функций, который для каждой продажи выведет:

Сумму всех продаж данного менеджера (агрегирующая оконная функция).

Номер продажи менеджера по порядку (ранжирующая оконная функция).

Сумму предыдущей продажи того же менеджера (функция сдвига).

В результирующем запросе должны быть такие колонки:

sale_id — идентификатор продажи

manager_id — идентификатор менеджера

sale_date — дата продажи

amount — сумма продажи

total_sales_by_manager — общая сумма продаж данного менеджера

sale_number — порядковый номер продажи менеджера по дате

prev_sale_amount — сумма предыдущей продажи менеджера

In [ ]:
sales = sales.sort_values(['manager_id', 'sale_date']).reset_index(drop=True)
sales['total_sales_by_manager'] = sales.groupby('manager_id')['amount'].transform('sum')
sales['sale_number'] = sales.groupby('manager_id').cumcount() + 1
sales['prev_sale_amount'] = sales.groupby('manager_id')['amount'].shift(1)
sales

,sale_id,manager_id,sale_date,amount,total_sales_by_manager,sale_number,prev_sale_amount
0,1,101,2023-01-01,100,550,1,NaN
1,2,101,2023-01-02,200,550,2,100.0
2,5,101,2023-01-03,250,550,3,200.0
3,3,102,2023-01-01,300,450,1,NaN
4,4,102,2023-01-03,150,450,2,300.0


8.1=====================================

In [ ]:

import pandas as pd

# Создаём DataFrame для таблицы clients с тестовыми данными
clients = pd.DataFrame({
    'client_id': [1, 2, 3, 4, 5, 6],
    'client_name': ['Эдуард', 'Анна', 'Петр', 'Ольга', 'Михаил', 'Елена']
}).astype({
    'client_id': 'int64',
    'client_name': 'string'
})

# Создаём DataFrame для таблицы orders с тестовыми данными
orders = pd.DataFrame({
    'order_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
    'client_id': [1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 4, 5, 6, 1, 1],
    'order_sum': [500.10, 344.20, 144.90, 44.70, 19.60, 244.80, 344.20,
                 192.00, 137.60, 344.20, 112.00, 117.30, 150.60, 500.10, 350.90],
    'order_date': [
        '2022-02-18 00:00:00', '2022-02-20 00:00:00', '2022-01-12 00:00:00',
        '2022-02-18 00:00:00', '2022-02-18 00:00:00', '2022-01-05 00:00:00',
        '2022-01-18 00:00:00', '2022-02-11 00:00:00', '2022-02-20 00:00:00',
        '2022-01-18 00:00:00', '2022-01-05 00:00:00', '2022-02-05 00:00:00',
        '2022-01-05 00:00:00', '2021-01-18 00:00:00', '2021-01-02 00:00:00'
    ],
    'order_state': [
        'Delivered', 'In Progress', 'In Progress', 'In Progress', 'In Progress',
        'In Progress', 'Delivered', 'Delivered', 'In Progress', 'In Progress',
        'Delivered', 'In Progress', 'In Progress', 'Delivered', 'Delivered'
    ]
})

# Преобразуем столбец order_date в тип datetime
orders['order_date'] = pd.to_datetime(orders['order_date'])

# Явно указываем типы данных для остальных столбцов таблицы orders
orders = orders.astype({
    'order_id': 'int64',
    'client_id': 'int64',
    'order_sum': 'float64',
    'order_state': 'string'
})

clients

,client_id,client_name
0,1,Эдуард
1,2,Анна
2,3,Петр
3,4,Ольга
4,5,Михаил
5,6,Елена


In [ ]:
orders.head()

,order_id,client_id,order_sum,order_date,order_state
0,1,1,500.1,2022-02-18,Delivered
1,2,1,344.2,2022-02-20,In Progress
2,3,1,144.9,2022-01-12,In Progress
3,4,2,44.7,2022-02-18,In Progress
4,5,2,19.6,2022-02-18,In Progress


Условие задачи
Напишите SQL-запрос, который выбирает имя клиента, оформившего заказ с максимальной суммой в январе 2022 года.

Максимальная сумма определяется среди всех заказов за январь 2022.
Если таких заказов несколько (несколько клиентов оформили заказ на одну и ту же максимальную сумму), то вывести всех этих клиентов.
Ожидаемые колонки в результирующем запросе:
client_name

In [ ]:
january_orders = orders[
    (orders['order_date'] >= '2022-01-01') &
    (orders['order_date'] < '2022-02-01')
]
max_sum = january_orders['order_sum'].max()
january_orders = january_orders[january_orders['order_sum'] == max_sum].merge(clients)[['client_name']]
january_orders

,client_name
0,Анна
1,Петр


In [ ]:
import pandas as pd

# Создаём словарь с данными
data = {
    'order_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],
    'client_id': [1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 4, 5, 6],
    'order_sum': [500.10, 344.20, 144.90, 44.70, 19.60, 244.80, 344.20, 192.00, 137.60, 344.20, 112.00, 117.30, 150.60],
    'order_date': [
        '2022-02-18 00:00:00',
        '2022-02-20 00:00:00',
        '2022-01-12 00:00:00',
        '2022-02-18 00:00:00',
        '2022-02-18 00:00:00',
        '2022-01-05 00:00:00',
        '2022-01-18 00:00:00',
        '2022-02-11 00:00:00',
        '2022-02-20 00:00:00',
        '2022-01-18 00:00:00',
        '2022-01-05 00:00:00',
        '2022-02-05 00:00:00',
        '2022-01-05 00:00:00'
    ],
    'order_state': [
        'Delivered', 'In Progress', 'In Progress', 'In Progress',
        'In Progress', 'In Progress', 'Delivered', 'Delivered',
        'In Progress', 'In Progress', 'Delivered', 'In Progress', 'In Progress'
    ]
}

# Создаём DataFrame
df = pd.DataFrame(data)

# Преобразуем столбец order_date в формат datetime
df['order_date'] = pd.to_datetime(df['order_date'])

df.head()

,order_id,client_id,order_sum,order_date,order_state
0,1,1,500.1,2022-02-18,Delivered
1,2,1,344.2,2022-02-20,In Progress
2,3,1,144.9,2022-01-12,In Progress
3,4,2,44.7,2022-02-18,In Progress
4,5,2,19.6,2022-02-18,In Progress


Условие задачи
 Напишите SQL-запрос, который выводит:

количество заказов со статусом Delivered
количество заказов со статусом In Progress
в разрезе даты (order_date), с сортировкой по дням в порядке возрастания.

Ожидаемые колонки в результирующем запросе:
order_date
count_delivered
count_in_progress
Отсортируйте результат по колонке
order_date

In [ ]:
df.groupby('order_date', as_index=False)\
  .agg(
      count_delivered=('order_state', lambda x:(x =='Delivered').sum()),
      count_in_progress=('order_state', lambda x:(x == 'In progerss').sum())
      ).sort_values(['order_date'])

,order_date,count_delivered,count_in_progress
0,2022-01-05,1,0
1,2022-01-12,0,0
2,2022-01-18,1,0
3,2022-02-05,0,0
4,2022-02-11,1,0
5,2022-02-18,1,0
6,2022-02-20,0,0


In [ ]:
df['is_delivered'] = (df['order_state'] == 'Delivered').astype(int)
df['in_progress'] = (df['order_state'] == 'In progress').astype(int)
df.groupby('order_date', as_index=False).agg(
    count_delivered=('is_delivered', 'sum'),
    count_in_progress=('in_progress', 'sum')
).sort_values(['order_date'])

,order_date,count_delivered,count_in_progress
0,2022-01-05,1,0
1,2022-01-12,0,0
2,2022-01-18,1,0
3,2022-02-05,0,0
4,2022-02-11,1,0
5,2022-02-18,1,0
6,2022-02-20,0,0


In [ ]:
result = (df.assign(
    is_delivered = (df['order_state'] == 'Delivered').astype(int),
    in_progress = (df['order_state'] == 'In progress').astype(int)
).groupby('order_date', as_index=False).agg(
    count_delivered=('is_delivered', 'sum'),
    count_in_progress=('in_progress', 'sum')
).sort_values('order_date'))
result

,order_date,count_delivered,count_in_progress
0,2022-01-05,1,0
1,2022-01-12,0,0
2,2022-01-18,1,0
3,2022-02-05,0,0
4,2022-02-11,1,0
5,2022-02-18,1,0
6,2022-02-20,0,0


In [ ]:
import pandas as pd

data = {
    'client_id': [1, 2, 3, 4, 5, 6],
    'client_name': ['Эдуард', 'Анна', 'Петр', 'Ольга', 'Михаил', 'Елена']
}

clients = pd.DataFrame(data)

clients

,client_id,client_name
0,1,Эдуард
1,2,Анна
2,3,Петр
3,4,Ольга
4,5,Михаил
5,6,Елена


In [ ]:
data = {
    'order_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],
    'client_id': [1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 4, 5, 6],
    'order_sum': [500.10, 344.20, 144.90, 44.70, 19.60, 244.80, 344.20, 192.00, 137.60, 344.20, 112.00, 117.30, 150.60],
    'order_date': [
        '2022-02-18 00:00:00',
        '2022-02-20 00:00:00',
        '2022-01-12 00:00:00',
        '2022-02-18 00:00:00',
        '2022-02-18 00:00:00',
        '2022-01-05 00:00:00',
        '2022-01-18 00:00:00',
        '2022-02-11 00:00:00',
        '2022-02-20 00:00:00',
        '2022-01-18 00:00:00',
        '2022-01-05 00:00:00',
        '2022-02-05 00:00:00',
        '2022-01-05 00:00:00'
    ],
    'order_state': [
        'Delivered', 'In Progress', 'In Progress', 'In Progress',
        'In Progress', 'In Progress', 'Delivered', 'Delivered',
        'In Progress', 'In Progress', 'Delivered', 'In Progress', 'In Progress'
    ]
}

orders = pd.DataFrame(data)
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders.head(5)

,order_id,client_id,order_sum,order_date,order_state
0,1,1,500.1,2022-02-18,Delivered
1,2,1,344.2,2022-02-20,In Progress
2,3,1,144.9,2022-01-12,In Progress
3,4,2,44.7,2022-02-18,In Progress
4,5,2,19.6,2022-02-18,In Progress


Напишите SQL-запрос, который выведет имя клиента, удовлетворяющего условиям:
- клиент совершил только один заказ
- этот заказ был сделан только на заданную дату — 2022-01-05
Ожидаемые колонки в результирующем запросе: client_name

In [ ]:
df = orders.groupby('client_id', as_index=False).agg(count_orders=('client_id', 'count'))
merge_df = df[df['count_orders'] == 1].merge(clients).merge(orders)
merge_df[merge_df['order_date'] == '2022-01-05'][['client_name']]

,client_name
0,Ольга
2,Елена


In [ ]:
single_order_clients = orders.groupby('client_id').filter(lambda x: len(x) == 1)
single_order_clients[single_order_clients['order_date'] == '2022-01-05'].merge(clients, on='client_id')[['client_name']]

,client_name
0,Ольга
1,Елена


In [ ]:
import pandas as pd
data = {
    'client_id': [60001, 60002, 60003, 70001, 70002],
    'total_balance': [1250.50, 0.00, 98500.75, 15320.00, 775.10],
    'update_ts': [
        '2024-05-10 08:15:22',
        '2024-05-10 09:20:30',
        '2024-05-11 11:45:10',
        '2024-05-12 14:05:00',
        '2024-05-13 16:30:45'
    ]
}

balance = pd.DataFrame(data)
balance['update_ts'] = pd.to_datetime(balance['update_ts'])
balance.head()


,client_id,total_balance,update_ts
0,60001,1250.50,2024-05-10 08:15:22
1,60002,0.00,2024-05-10 09:20:30
2,60003,98500.75,2024-05-11 11:45:10
3,70001,15320.00,2024-05-12 14:05:00
4,70002,775.10,2024-05-13 16:30:45


In [ ]:
data = {
    'client_id': [60001, 60002, 60003, 70001, 70002],
    'fio': [
        'Петров Петр Петрович',
        'Сидорова Анна',
        'John Smith',
        'Chen Wei',
        'Александр Сергеевич'
    ]
}
fio = pd.DataFrame(data)
fio.head()

,client_id,fio
0,60001,Петров Петр Петрович
1,60002,Сидорова Анна
2,60003,John Smith
3,70001,Chen Wei
4,70002,Александр Сергеевич


In [ ]:
data = {
    'date': [
        '2022-12-01',
        '2023-01-15',
        '2023-03-18',
        '2024-05-10',
        '2024-05-11',
        '2024-05-12',
        '2024-05-12',
        '2024-05-13',
        '2024-05-14',
        '2024-05-13',
        '2024-05-14',
        '2024-04-15',
        '2024-04-20',
        '2024-04-10'
    ],
    'client_id': [
        60001, 60001, 60002, 60002, 60003, 60003, 60003,
        70001, 70001, 70002, 70002, 60002, 60003, 70001
    ],
    'client_activity_type_id': [
        1, 2, 3, 1, 2, 4, 6, 1, 5, 1, 3, 1, 2, 3
    ]
}
client_activity = pd.DataFrame(data)
client_activity['date'] = pd.to_datetime(client_activity['date'])
client_activity.head()


,date,client_id,client_activity_type_id
0,2022-12-01,60001,1
1,2023-01-15,60001,2
2,2023-03-18,60002,3
3,2024-05-10,60002,1
4,2024-05-11,60003,2


Есть таблица balance с данными по балансам клиентов:
- client_id – идентификатор клиента  
- total_balance – текущий баланс
- update_ts – время обновления информации  
Таблица fio с именами клиентов:
- client_id – идентификатор клиента
- fio – ФИО   
И таблица client_activity, в которой хранятся данные по различным видам активности клиентов на сайте:
- date – дата
- client_id – идентификатор клиента
- client_activity_type_id– тип активности     
❗ Нужно вывести ФИО клиента/клиентов с максимальным балансом за текущий месяц.
Текущий месяц определяется по максимальной дате в таблице balance.
Возможна ситуация, когда условие выполняют несколько клиентов, тогда выводим всех.
Ожидаемые колонки в результирующем запросе:
fio,
total_balance

In [ ]:
last_date = balance['update_ts'].max()
curr_month_mask = (balance['update_ts'].dt.month == last_date.month) & \
                  (balance['update_ts'].dt.year == last_date.year)
actual = (
    balance[curr_month_mask]
    .sort_values('update_ts', ascending=False)
    .drop_duplicates('client_id')
)
max_val = actual['total_balance'].max()
actual[actual['total_balance'] == max_val].merge(fio, on='client_id')[['fio', 'total_balance']]

,fio,total_balance
0,John Smith,98500.75


Нужно найти количество уникальных клиентов, у которых была хотя бы одна активность:

и в текущем месяце (по данным активности),
и в прошлом месяце.

In [ ]:
import numpy as np
curr_month_start = pd.to_datetime(balance['update_ts'].max()).replace(day=1)
prev_month_start = curr_month_start - pd.DateOffset(months=1)
id_curr = client_activity[client_activity['date'] >= curr_month_start]['client_id'].unique()
mask_prev = (client_activity['date'] >= prev_month_start) & (client_activity['date'] < curr_month_start)
id_prev = client_activity[mask_prev]['client_id'].unique()
pd.DataFrame({'active_client': [len(np.intersect1d(id_curr, id_prev))]})

,active_client
0,3


Точность прогноза длительности сборки. У вас есть таблицы маршрутных листов (workflows) и сегментов (segments).

Нужно посчитать, насколько в среднем совпадает плановая длительность сегмента типа assembly с фактической.

Учитываются только выполненные маршрутные листы (status = 'completed') за последнюю неделю относительно текущей даты.

Ожидаемая колонка в результате:

avg_diff_minutes — средняя разница (в минутах).

In [ ]:
import pandas as pd
data = {
    'id': [1, 2, 3, 4],
    'status': ['completed', 'completed', 'canceled', 'completed'],
    'performer_uuid': ['a1', 'a2', 'a3', 'a4'],
    'created_at': [
        '2026-03-22 07:54:56',
        '2026-03-24 07:54:56',
        '2026-03-25 07:54:56',
        '2026-03-17 07:54:56'
    ],
    'updated_at': [
        '2026-03-22 07:54:56',
        '2026-03-24 07:54:56',
        '2026-03-25 07:54:56',
        '2026-03-17 07:54:56'
    ]
}

workflows = pd.DataFrame(data)
workflows['created_at'] = pd.to_datetime(workflows['created_at'])
workflows['updated_at'] = pd.to_datetime(workflows['updated_at'])
workflows.head()



,id,status,performer_uuid,created_at,updated_at
0,1,completed,a1,2026-03-22 07:54:56,2026-03-22 07:54:56
1,2,completed,a2,2026-03-24 07:54:56,2026-03-24 07:54:56
2,3,canceled,a3,2026-03-25 07:54:56,2026-03-25 07:54:56
3,4,completed,a4,2026-03-17 07:54:56,2026-03-17 07:54:56


In [ ]:
data = {
    'id': [1, 2, 3, 4, 5],
    'workflow_id': [1, 1, 2, 3, 4],
    'segment_type': ['assembly', 'delivery', 'assembly', 'assembly', 'assembly'],
    'plan_started_at': [
        '2022-01-01 10:00:00',
        '2022-01-01 10:40:00',
        '2022-01-02 09:00:00',
        '2022-01-03 12:00:00',
        '2022-01-04 14:00:00'
    ],
    'plan_ended_at': [
        '2022-01-01 10:30:00',
        '2022-01-01 11:00:00',
        '2022-01-02 09:20:00',
        '2022-01-03 12:25:00',
        '2022-01-04 14:30:00'
    ],
    'fact_started_at': [
        '2022-01-01 10:00:00',
        '2022-01-01 10:45:00',
        '2022-01-02 09:05:00',
        '2022-01-03 12:05:00',
        '2022-01-04 14:05:00'
    ],
    'fact_ended_at': [
        '2022-01-01 10:40:00',
        '2022-01-01 11:10:00',
        '2022-01-02 09:25:00',
        '2022-01-03 12:35:00',
        '2022-01-04 14:45:00'
    ]
}

segments = pd.DataFrame(data)

datetime_columns = ['plan_started_at', 'plan_ended_at', 'fact_started_at', 'fact_ended_at']
for col in datetime_columns:
    segments[col] = pd.to_datetime(segments[col])
segments.head()


,id,workflow_id,segment_type,plan_started_at,plan_ended_at,fact_started_at,fact_ended_at
0,1,1,assembly,2022-01-01 10:00:00,2022-01-01 10:30:00,2022-01-01 10:00:00,2022-01-01 10:40:00
1,2,1,delivery,2022-01-01 10:40:00,2022-01-01 11:00:00,2022-01-01 10:45:00,2022-01-01 11:10:00
2,3,2,assembly,2022-01-02 09:00:00,2022-01-02 09:20:00,2022-01-02 09:05:00,2022-01-02 09:25:00
3,4,3,assembly,2022-01-03 12:00:00,2022-01-03 12:25:00,2022-01-03 12:05:00,2022-01-03 12:35:00
4,5,4,assembly,2022-01-04 14:00:00,2022-01-04 14:30:00,2022-01-04 14:05:00,2022-01-04 14:45:00


In [ ]:
workflows_f = workflows.loc[(workflows['updated_at'] >= workflows.updated_at.max() - pd.Timedelta(days=7)) & (workflows['status'] == 'completed')]['id']
segments_f = segments[segments['segment_type'] == 'assembly'].merge(workflows_f, left_on='workflow_id', right_on='id')
plan_dur = (segments_f['plan_ended_at'] - segments_f['plan_started_at']).dt.total_seconds()
fact_dur = (segments_f['fact_ended_at'] - segments_f['fact_started_at']).dt.total_seconds()
avg_diff_minutes = (plan_dur - fact_dur).mean() / 60
pd.DataFrame({'avg_diff_minutes': [(plan_dur - fact_dur).mean() / 60]})

,avg_diff_minutes
0,-5.0


Даны таблицы маршрутных листов WORKFLOWS и их сегментов SEGMENTS.

Нужно посчитать, сколько в среднем времени уходит у партнёра на один маршрутный лист, сгруппировав по партнёру (performer_uuid). Под временем на маршрутный лист понимается:

начало: минимальное fact_started_at среди всех сегментов данного листа;

конец:

если у листа есть завершённые сегменты — максимальное fact_ended_at среди них;

если лист отменён и у “текущего” сегмента fact_ended_at не задан — используем workflows.updated_at (момент смены статуса).

Ожидаемые колонки:
partner_uuid,
avg_workflow_minutes — среднее (в минутах) время на маршрутный лист по каждому партнёру.

In [ ]:
import pandas as pd
data = {
    'id': [1, 2, 3, 4],
    'status': ['completed', 'canceled', 'completed', 'canceled'],
    'performer_uuid': ['p1', 'p1', 'p2', 'p3'],
    'created_at': [
        '2022-01-01 09:00:00',
        '2022-01-02 10:00:00',
        '2022-01-03 08:00:00',
        '2022-01-04 07:45:00'
    ],
    'updated_at': [
        '2022-01-01 12:30:00',
        '2022-01-02 11:10:00',
        '2022-01-03 10:15:00',
        '2022-01-04 08:05:00'
    ]
}
workflows = pd.DataFrame(data)
workflows.head()

,id,status,performer_uuid,created_at,updated_at
0,1,completed,p1,2022-01-01 09:00:00,2022-01-01 12:30:00
1,2,canceled,p1,2022-01-02 10:00:00,2022-01-02 11:10:00
2,3,completed,p2,2022-01-03 08:00:00,2022-01-03 10:15:00
3,4,canceled,p3,2022-01-04 07:45:00,2022-01-04 08:05:00


In [ ]:
import pandas as pd
import numpy as np
data = {
    'id': [101, 102, 103, 201, 202, 301, 302, 401],
    'created_at': [
        '2022-01-01 09:05:00',
        '2022-01-01 10:00:00',
        '2022-01-01 11:30:00',
        '2022-01-02 10:00:00',
        '2022-01-02 10:20:00',
        '2022-01-03 08:00:00',
        '2022-01-03 09:00:00',
        '2022-01-04 07:45:00'
    ],
    'updated_at': [
        '2022-01-01 09:50:00',
        '2022-01-01 10:40:00',
        '2022-01-01 12:25:00',
        '2022-01-02 10:20:00',
        '2022-01-02 11:10:00',
        '2022-01-03 08:50:00',
        '2022-01-03 10:10:00',
        '2022-01-04 08:05:00'
    ],
    'workflow_id': [1, 1, 1, 2, 2, 3, 3, 4],
    'segment_type': ['assembly', 'delivery', 'handoff', 'assembly', 'delivery', 'assembly', 'delivery', 'assembly'],
    'order_n': [1, 2, 3, 1, 2, 1, 2, 1],
    'plan_started_at': [
        '2022-01-01 09:10:00',
        '2022-01-01 10:00:00',
        '2022-01-01 11:35:00',
        '2022-01-02 10:00:00',
        '2022-01-02 10:20:00',
        '2022-01-03 08:05:00',
        '2022-01-03 09:05:00',
        '2022-01-04 07:45:00'
    ],
    'plan_ended_at': [
        '2022-01-01 09:40:00',
        '2022-01-01 10:30:00',
        '2022-01-01 12:25:00',
        '2022-01-02 10:15:00',
        '2022-01-02 11:20:00',
        '2022-01-03 08:40:00',
        '2022-01-03 10:00:00',
        '2022-01-04 08:10:00'
    ],
    'fact_started_at': [
        '2022-01-01 09:10:00',
        '2022-01-01 10:05:00',
        '2022-01-01 11:40:00',
        '2022-01-02 10:05:00',
        '2022-01-02 10:22:00',
        '2022-01-03 08:10:00',
        '2022-01-03 09:15:00',
        '2022-01-04 07:50:00'
    ],
    'fact_ended_at': [
        '2022-01-01 09:45:00',
        '2022-01-01 10:35:00',
        '2022-01-01 12:20:00',
        '2022-01-02 10:18:00',
        np.nan,  # NULL для строки 202
        '2022-01-03 08:45:00',
        '2022-01-03 10:10:00',
        np.nan   # NULL для строки 401
    ]
}
segments = pd.DataFrame(data)
date_columns = ['created_at', 'updated_at', 'plan_started_at', 'plan_ended_at', 'fact_started_at', 'fact_ended_at']
for col in date_columns:
    segments[col] = pd.to_datetime(segments[col])
segments



,id,created_at,updated_at,workflow_id,segment_type,order_n,plan_started_at,plan_ended_at,fact_started_at,fact_ended_at
0,101,2022-01-01 09:05:00,2022-01-01 09:50:00,1,assembly,1,2022-01-01 09:10:00,2022-01-01 09:40:00,2022-01-01 09:10:00,2022-01-01 09:45:00
1,102,2022-01-01 10:00:00,2022-01-01 10:40:00,1,delivery,2,2022-01-01 10:00:00,2022-01-01 10:30:00,2022-01-01 10:05:00,2022-01-01 10:35:00
2,103,2022-01-01 11:30:00,2022-01-01 12:25:00,1,handoff,3,2022-01-01 11:35:00,2022-01-01 12:25:00,2022-01-01 11:40:00,2022-01-01 12:20:00
3,201,2022-01-02 10:00:00,2022-01-02 10:20:00,2,assembly,1,2022-01-02 10:00:00,2022-01-02 10:15:00,2022-01-02 10:05:00,2022-01-02 10:18:00
4,202,2022-01-02 10:20:00,2022-01-02 11:10:00,2,delivery,2,2022-01-02 10:20:00,2022-01-02 11:20:00,2022-01-02 10:22:00,NaT
5,301,2022-01-03 08:00:00,2022-01-03 08:50:00,3,assembly,1,2022-01-03 08:05:00,2022-01-03 08:40:00,2022-01-03 08:10:00,2022-01-03 08:45:00
6,302,2022-01-03 09:00:00,2022-01-03 10:10:00,3,delivery,2,2022-01-03 09:05:00,2022-01-03 10:00:00,2022-01-03 09:15:00,2022-01-03 10:10:00
7,401,2022-01-04 07:45:00,2022-01-04 08:05:00,4,assembly,1,2022-01-04 07:45:00,2022-01-04 08:10:00,2022-01-04 07:50:00,NaT


In [ ]:
seg_mnmx = segments.groupby('workflow_id', as_index=False).agg(start=('fact_started_at', 'min'), end=('fact_ended_at', 'max'))
seg_work = seg_mnmx.merge(workflows, left_on='workflow_id', right_on='id')
seg_work['workflow_minutes'] = (seg_work['end'].fillna(seg_work['updated_at']) - seg_work['start']).dt.total_seconds() / 60
seg_work.groupby('performer_uuid', as_index=False).agg(avg_workflow_minutes=('workflow_minutes', 'mean'))

,performer_uuid,avg_workflow_minutes
0,p1,101.5
1,p2,120.0
2,p3,15.0


In [ ]:
workflows.merge(segments, left_on='id', right_on='workflow_id')\
.groupby('workflow_id', as_index=False)\
.agg(
    start=('fact_started_at', 'min'),
    end=('fact_ended_at', 'max'),
    performer_uuid=('performer_uuid', 'first'), # Берем исполнителя из группы
    updated_at=('updated_at_x', 'first')
)\
.assign(
    end=lambda x: x['end'].fillna(x['updated_at']),
    workflow_minutes=lambda x: (x['end'] - x['start']).dt.total_seconds() / 60
)\
.groupby('performer_uuid', as_index=False)\
.agg(avg_workflow_minutes=('workflow_minutes', 'mean'))

,performer_uuid,avg_workflow_minutes
0,p1,101.5
1,p2,120.0
2,p3,15.0


На основании таблицы с данными о пользователях, заказах и городах необходимо вывести топ 5% пользователей по количеству совершенных покупок (заказов) в разрезе каждого города. То есть для каждого города нужно определить своих топ 5% пользователей с наибольшим количеством заказов.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Для воспроизводимости результатов (можно убрать)
random.seed(42)
np.random.seed(42)

# Параметры
n_orders = 250
n_users = 50
n_cities = 5
days_back = 365

# Генерация данных
user_id = np.random.randint(1, n_users + 1, n_orders)
order_id = np.arange(1, n_orders + 1)
city_id = np.random.randint(1, n_cities + 1, n_orders)

# Даты заказов за последний год
start_date = datetime.now() - timedelta(days=days_back)
order_date = [start_date + timedelta(days=random.randint(0, days_back))
              for _ in range(n_orders)]

# Суммы заказов
amount = np.round(np.random.uniform(10.0, 1010.0, n_orders), 2)

# Создаём DataFrame
df = pd.DataFrame({
    'user_id': user_id,
    'order_id': order_id,
    'city_id': city_id,
    'order_date': order_date,
    'amount': amount
})

# Приводим типы (по желанию)
df['user_id'] = df['user_id'].astype('int32')
df['order_id'] = df['order_id'].astype('int32')
df['city_id'] = df['city_id'].astype('int32')
df['amount'] = df['amount'].astype('float64')

df.head()

,user_id,order_id,city_id,order_date,amount
0,39,1,4,2026-02-20 08:29:02.601327,693.01
1,29,2,3,2025-05-26 08:29:02.601327,81.19
2,15,3,4,2025-04-11 08:29:02.601327,328.98
3,43,4,3,2025-08-17 08:29:02.601327,854.88
4,8,5,2,2025-08-02 08:29:02.601327,33.27


In [ ]:
df_1 = df.groupby(['city_id', 'user_id'], as_index=False).agg(order_count=('order_id', 'count'))
df_1['count_users'] = df_1.groupby('city_id')['user_id'].transform('nunique')
df_1['user_rank'] = df_1.groupby('city_id')['order_count'].rank(method='dense', ascending=False)
df_1[df_1['user_rank'] / df_1['count_users'] <= 0.05]

,city_id,user_id,order_count,count_users,user_rank
18,1,28,3,34,1.0
48,2,26,4,27,1.0
68,3,15,4,29,1.0
116,4,37,4,34,1.0
117,4,39,4,34,1.0
120,4,44,4,34,1.0
129,5,9,5,28,1.0


Есть две таблицы: список городов с указанием страны и таблица продаж по городам.
Для каждой страны необходимо определить ТОП-5 городов по суммарной выручке.
Выручка города — это сумма значений income по всем продажам в этом городе.
Если в стране меньше 5 городов, нужно вывести все доступные города.
Ожидаемые колонки в результирующем запросе:
- country_name  
- city  
- total_income  
Отсортируйте результат по колонкам   
- country_name  
- total_income (по убыванию)  

In [ ]:
import pandas as pd
data = {
    'sale_date': [
        '2025-04-01 10:00:00',
        '2025-04-01 12:30:00',
        '2025-04-02 09:15:00',
        '2025-04-02 14:20:00',
        '2025-04-03 11:10:00',
        '2025-04-03 16:45:00',
        '2025-04-04 13:00:00',
        '2025-04-01 10:00:00',
        '2025-04-01 11:30:00',
        '2025-04-02 15:00:00',
        '2025-04-02 17:20:00',
        '2025-04-03 12:10:00',
        '2025-04-01 09:00:00',
        '2025-04-02 10:30:00',
        '2025-04-03 14:00:00'
    ],
    'city': [
        'Moscow',
        'Moscow',
        'Saint Petersburg',
        'Kazan',
        'Novosibirsk',
        'Yekaterinburg',
        'Sochi',
        'Berlin',
        'Munich',
        'Hamburg',
        'Cologne',
        'Frankfurt',
        'Paris',
        'Lyon',
        'Marseille'
    ],
    'income': [
        1200.50,
        800.00,
        950.00,
        400.00,
        300.00,
        500.00,
        700.00,
        1100.00,
        900.00,
        650.00,
        300.00,
        750.00,
        2000.00,
        850.00,
        600.00
    ]
}
sales = pd.DataFrame(data)
sales['sale_date'] = pd.to_datetime(sales['sale_date'])
sales.head()

,sale_date,city,income
0,2025-04-01 10:00:00,Moscow,1200.5
1,2025-04-01 12:30:00,Moscow,800.0
2,2025-04-02 09:15:00,Saint Petersburg,950.0
3,2025-04-02 14:20:00,Kazan,400.0
4,2025-04-03 11:10:00,Novosibirsk,300.0


In [ ]:
data = {
    'country_name': [
        'Russia', 'Russia', 'Russia', 'Russia', 'Russia', 'Russia',
        'Germany', 'Germany', 'Germany', 'Germany', 'Germany',
        'France', 'France', 'France'
    ],
    'city': [
        'Moscow', 'Saint Petersburg', 'Kazan', 'Novosibirsk', 'Yekaterinburg', 'Sochi',
        'Berlin', 'Munich', 'Hamburg', 'Cologne', 'Frankfurt',
        'Paris', 'Lyon', 'Marseille'
    ]
}

country = pd.DataFrame(data)
country.head()


,country_name,city
0,Russia,Moscow
1,Russia,Saint Petersburg
2,Russia,Kazan
3,Russia,Novosibirsk
4,Russia,Yekaterinburg


In [ ]:
df = sales.groupby('city', as_index=False).agg(total_income=('income', 'sum')).merge(country, how='left')
df['city_rank'] = df.groupby('country_name')['total_income'].rank(method='dense', ascending=False)
df[df['city_rank'] <= 5].sort_values(by=['country_name', 'total_income'], ascending=[True, False])[['country_name', 'city', 'total_income']]

,country_name,city,total_income
10,France,Paris,2000.0
5,France,Lyon,850.0
6,France,Marseille,600.0
0,Germany,Berlin,1100.0
8,Germany,Munich,900.0
2,Germany,Frankfurt,750.0
3,Germany,Hamburg,650.0
1,Germany,Cologne,300.0
7,Russia,Moscow,2000.5
11,Russia,Saint Petersburg,950.0


1. Таблица ads (Рекламные объявления):
- ad_id (INT, PRIMARY KEY) - идентификатор объявления
- campaign_id (INT, FOREIGN KEY) - идентификатор рекламной кампании
- status (VARCHAR) - статус объявления  
2. Таблица events (События):
- event_id (INT, PRIMARY KEY) - идентификатор события
- ad_id (INT, FOREIGN KEY) - идентификатор объявления
- source (VARCHAR) - источник события
- event_type (VARCHAR) - тип события
- event_date (DATE) - дата события
- event_hour (INT) - час события  
Необходимо получить количество событий, сгруппированных по следующим признакам:
- campaign_id — идентификатор рекламной кампании
- event_type — тип события
Иными словами, нужно определить, сколько событий каждого типа произошло в рамках каждой рекламной кампании.
В результирующем запросе будет три колонки:
- campaign_id	INT	Идентификатор рекламной кампании
- event_type	VARCHAR	Тип события (например, click, view, purchase)
- events_count	INT	Количество событий данного типа в рамках кампании


In [ ]:
import pandas as pd
data = {
    'event_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'ad_id': [1, 1, 2, 2, 3, 3, 4, 4, 5, 5],
    'source': ['web', 'web', 'mobile', 'web', 'web', 'mobile', 'web', 'mobile', 'web', 'web'],
    'event_type': ['impression', 'click', 'impression', 'impression', 'click',
                  'impression', 'impression', 'click', 'impression', 'impression'],
    'event_date': ['2023-01-01', '2023-01-01', '2023-01-01', '2023-01-01',
                   '2023-01-01', '2023-01-01', '2023-01-02', '2023-01-02',
                   '2023-01-02', '2023-01-02'],
    'event_hour': [10, 11, 12, 13, 14, 15, 9, 10, 11, 12]
}
events = pd.DataFrame(data)
events['event_date'] = pd.to_datetime(events['event_date'])
events.head()

,event_id,ad_id,source,event_type,event_date,event_hour
0,1,1,web,impression,2023-01-01,10
1,2,1,web,click,2023-01-01,11
2,3,2,mobile,impression,2023-01-01,12
3,4,2,web,impression,2023-01-01,13
4,5,3,web,click,2023-01-01,14


In [ ]:
import pandas as pd
data = {
    'ad_id': [1, 2, 3, 4, 5],
    'campaign_id': [101, 101, 102, 102, 103],
    'status': ['active', 'paused', 'active', 'active', 'paused']
}
ads = pd.DataFrame(data)
ads.head()


,ad_id,campaign_id,status
0,1,101,active
1,2,101,paused
2,3,102,active
3,4,102,active
4,5,103,paused


In [ ]:
gr_events = events.groupby(['ad_id', 'event_type'], as_index=False).agg(total_events=('event_type', 'count'))
gr_events.merge(ads, how='left').groupby(['campaign_id', 'event_type'], as_index=False).agg(event_count=('total_events', 'sum'))

,campaign_id,event_type,event_count
0,101,click,1
1,101,impression,3
2,102,click,2
3,102,impression,2
4,103,impression,2


Необходимо рассчитать CTR (Click-Through Rate) — показатель кликабельности рекламных объявлений — для каждой рекламной кампании.

In [ ]:
cl_imp = events.groupby('ad_id')['event_type'].value_counts().unstack(fill_value=0).reset_index()
result = cl_imp.merge(ads, how='left').groupby('campaign_id', as_index=False).agg(clicks=('click', 'sum'), impressions=('impression', 'sum'))
result['CTR'] = (result['clicks'] / result['impressions']).round(4)
result

,campaign_id,clicks,impressions,CTR
0,101,1,3,0.3333
1,102,2,2,1.0000
2,103,0,2,0.0000


 Дана таблица marks с оценками студентов по курсам (шкала 0–100). Таблица содержит поля: student_name, course_name, mark. Нужно вывести топ-5 лучших студентов в каждом курсе.
Правила ранжирования:
- Лучшие — с более высокой оценкой mark
- Если оценки равны, выше в рейтинге должен быть студент с именем, которое меньше по алфавиту (student_name по возрастанию)
- В результате выведите все исходные столбцы таблицы, добавив столбец rating_place — место студента в рейтинге своего курса (1 — лучший).  
Ожидаемые колонки в результирующем запросе:
- student_name
- course_name
- mark
- rating_place  
Отсортируйте результат по колонкам
- course_name (по возрастанию)
- rating_place (по возрастанию)

In [ ]:
import pandas as pd
data = {
    'student_name': [
        'Ivan Petrov', 'Anna Smirnova', 'Pavel Ivanov', 'Olga Sokolova',
        'Dmitry Kuznetsov', 'Sergey Volkov', 'Elena Morozova', 'Anna Smirnova',
        'Ivan Petrov', 'Pavel Ivanov', 'Elena Morozova', 'Olga Sokolova',
        'Dmitry Kuznetsov', 'Sergey Volkov', 'Ivan Petrov', 'Anna Smirnova',
        'Pavel Ivanov', 'Olga Sokolova', 'Dmitry Kuznetsov', 'Sergey Volkov',
        'Elena Morozova'
    ],
    'course_name': [
        'Algorithms', 'Algorithms', 'Algorithms', 'Algorithms',
        'Algorithms', 'Algorithms', 'Algorithms', 'Databases',
        'Databases', 'Databases', 'Databases', 'Databases',
        'Databases', 'Databases', 'Python', 'Python',
        'Python', 'Python', 'Python', 'Python', 'Python'
    ],
    'mark': [
        95, 98, 90, 88, 91, 91, 84, 100,
        92, 92, 85, 89, 76, 94, 78, 88,
        88, 91, 91, 83, 95
    ]
}

marks = pd.DataFrame(data)
marks.head()


,student_name,course_name,mark
0,Ivan Petrov,Algorithms,95
1,Anna Smirnova,Algorithms,98
2,Pavel Ivanov,Algorithms,90
3,Olga Sokolova,Algorithms,88
4,Dmitry Kuznetsov,Algorithms,91


In [ ]:
df = marks.sort_values(by=['course_name', 'mark', 'student_name'], ascending=[True,  False, True])
df['rating_place'] = df.groupby('course_name').cumcount() + 1
df[df['rating_place'] <= 5].sort_values(by=['course_name', 'rating_place'])

,student_name,course_name,mark,rating_place
1,Anna Smirnova,Algorithms,98,1
0,Ivan Petrov,Algorithms,95,2
4,Dmitry Kuznetsov,Algorithms,91,3
5,Sergey Volkov,Algorithms,91,4
2,Pavel Ivanov,Algorithms,90,5
7,Anna Smirnova,Databases,100,1
13,Sergey Volkov,Databases,94,2
8,Ivan Petrov,Databases,92,3
9,Pavel Ivanov,Databases,92,4
11,Olga Sokolova,Databases,89,5


Напишите SQL-запрос, который рассчитает долю (в процентах) каждого платёжного метода за день.

📌 Особенности:

- Для каждого дня необходимо посчитать сумму оборота по всем платёжным методам.
- Для каждого метода вычислить его долю = (оборот метода / общий оборот за день) * 100.
- Результат вывести в разрезе дата — метод оплаты — доля (%), округленная до двух знаков после запятой.
- Колонки в результирующем запросе: date, pay_method, percentage

In [ ]:
import pandas as pd
data = {
    'date': ['2023-01-02', '2023-01-02', '2023-01-02',
            '2023-01-03', '2023-01-03', '2023-01-03'],
    'pay_method': ['СБП', 'Сберпей', 'Кошелек',
                  'СБП', 'Сберпей', 'Кошелек'],
    'turnover': [500000.00, 1700000.00, 750000.00,
                600000.00, 2300000.00, 450000.00]
}
finance = pd.DataFrame(data)
finance['date'] = pd.to_datetime(finance['date'])
finance.head()

,date,pay_method,turnover
0,2023-01-02,СБП,500000.0
1,2023-01-02,Сберпей,1700000.0
2,2023-01-02,Кошелек,750000.0
3,2023-01-03,СБП,600000.0
4,2023-01-03,Сберпей,2300000.0


In [ ]:
df = finance.groupby(['date', 'pay_method'], as_index=False)['turnover'].sum()
df['percentage'] = (df['turnover'] / df.groupby('date')['turnover'].transform('sum')*100).round(2)
df[['date', 'pay_method', 'percentage']]

,date,pay_method,percentage
0,2023-01-02,Кошелек,25.42
1,2023-01-02,СБП,16.95
2,2023-01-02,Сберпей,57.63
3,2023-01-03,Кошелек,13.43
4,2023-01-03,СБП,17.91
5,2023-01-03,Сберпей,68.66


В сети розничных магазинов L-Sport в январе 2024 года был запущен новый метод оплаты — «СБП».
Необходимо написать SQL-запрос, который для каждого магазина рассчитает и сравнит:
- среднюю стоимость чека до введения метода оплаты «СБП»
- среднюю стоимость чека после введения метода оплаты «СБП»
- Показатели, перечисленные выше, округлить до 2 знаков после запятой
- Результат отсортировать по store_id  
📌 Особенности:
- Средняя стоимость чека =общая сумма всех чеков / количество чеков.
- Дата запуска метода «СБП» — январь 2024.
- Нужно сгруппировать результат по магазинам (store_id, store_name).  
Перечень полей в результирующем запросе:
- store_id — идентификатор магазина
- store_name — название магазина
- avg_receipt_before — средняя стоимость чека до появления метода оплаты «СБП»
- avg_receipt_after — средняя стоимость чека после появления метода оплаты «СБП»

In [ ]:
import pandas as pd
data = [
    [1, 'Outlet', 'Магазин 1', 101],
    [2, 'Concept', 'Магазин 2', 102],
    [3, 'Outlet', 'Магазин 3', 103]
]
stores = pd.DataFrame(data, columns=['store_id', 'store_format', 'store_name', 'manager_id'])


In [ ]:
receipts = pd.DataFrame({
    'receipt_id': [1001, 1002, 1003, 1004, 1005],
    'receipt_dttm': pd.to_datetime(['2023-12-15', '2023-12-20', '2024-01-10', '2024-01-15', '2024-01-20']),
    'salesman_name': ['Иванов Иван', 'Петров Петр', 'Сидоров Сидор', 'Иванов Иван', 'Петров Петр'],
    'payment_type_id': [1, 2, 3, 3, 1]
})

In [ ]:
sales = pd.DataFrame({
    'store_id': pd.Series([1, 1, 1, 2, 2, 3], dtype='int8'),
    'receipt_id': pd.Series([1001, 1001, 1004, 1002, 1005, 1003], dtype='int16'),
    'receipt_ln': pd.Series([1, 2, 1, 1, 1, 1], dtype='int8'),
    'product_id': pd.Series([101, 102, 105, 103, 106, 104], dtype='int16'),
    'price': pd.Series([500.00, 300.00, 1200.00, 1000.00, 700.00, 800.00], dtype='float32'),
    'qty': pd.Series([2.00, 1.00, 1.00, 1.00, 2.00, 3.00], dtype='float32')
})


In [ ]:
stores.head()

,store_id,store_format,store_name,manager_id
0,1,Outlet,Магазин 1,101
1,2,Concept,Магазин 2,102
2,3,Outlet,Магазин 3,103


In [ ]:
receipts.head()

,receipt_id,receipt_dttm,salesman_name,payment_type_id
0,1001,2023-12-15,Иванов Иван,1
1,1002,2023-12-20,Петров Петр,2
2,1003,2024-01-10,Сидоров Сидор,3
3,1004,2024-01-15,Иванов Иван,3
4,1005,2024-01-20,Петров Петр,1


In [ ]:
sales.head()

,store_id,receipt_id,receipt_ln,product_id,price,qty
0,1,1001,1,101,500.0,2.0
1,1,1001,2,102,300.0,1.0
2,1,1004,1,105,1200.0,1.0
3,2,1002,1,103,1000.0,1.0
4,2,1005,1,106,700.0,2.0


In [ ]:
df = sales.merge(receipts).merge(stores)
df['cost'] = df['price'] * df['qty']
df = df.groupby(['store_id', 'store_name', 'receipt_id', 'receipt_dttm'], as_index=False).agg(total_receipt=('cost', 'sum'))
df['rec_before'] = df['total_receipt'].where(df['receipt_dttm'] < '2024-01-01')
df['rec_after'] = df['total_receipt'].where(df['receipt_dttm'] >= '2024-01-01')
df.groupby(['store_id', 'store_name'], as_index=False).agg(avg_receipt_before=('rec_before', 'mean'), avg_receipt_after=('rec_after', 'mean'))

,store_id,store_name,avg_receipt_before,avg_receipt_after
0,1,Магазин 1,1300.0,1200.0
1,2,Магазин 2,1000.0,1400.0
2,3,Магазин 3,NaN,2400.0


In [ ]:
df = (sales.merge(receipts[['receipt_id', 'store_id', 'receipt_dttm']])
           .merge(stores[['store_id', 'store_name']]))
df = (df.assign(cost = df['price'] * df['qty'])
        .groupby(['store_id', 'store_name', 'receipt_id', 'receipt_dttm'], as_index=False)
        .agg(total_receipt=('cost', 'sum')))
res = (df.assign(
            before = df['total_receipt'].where(df['receipt_dttm'] < '2024-01-01'),
            after  = df['total_receipt'].where(df['receipt_dttm'] >= '2024-01-01')
        )
        .groupby(['store_id', 'store_name'], as_index=False)
        .agg(avg_before=('before', 'mean'), avg_after=('after', 'mean'))
        .round(2))

Есть две таблицы:
- Users (Пользователи): содержит информацию о пользователях и дате их регистрации.  
- Subscriptions (Подписки): хранит данные о подписках пользователей, включая даты начала и окончания действия подписки.    
Необходимо написать SQL-запрос, который для каждого года в диапазоне с 2019 по 2024 выведет:
- Год.
- Количество новых пользователей, зарегистрировавшихся в этом году.
- Количество активных подписчиков в этом году (то есть тех, у кого в данном году подписка действовала хотя бы один день).
- Результат должен быть отсортирован по году в порядке убывания.

In [ ]:
import pandas as pd
users = pd.DataFrame({
    'user_id': [1, 2, 3, 4, 5, 6],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank'],
    'signup_date': pd.to_datetime(['2019-03-15', '2020-06-10', '2021-01-20',
                                   '2022-11-05', '2023-08-30', '2024-02-14'])
})

In [ ]:
subscriptions = pd.DataFrame({
    'subscription_id': [1, 2, 3, 4, 5, 6],
    'user_id': [1, 2, 3, 4, 5, 6],
    'start_date': pd.to_datetime(['2019-03-16', '2020-06-10', '2021-01-20',
                                 '2022-11-05', '2023-08-30', '2024-02-14']),
    'end_date': pd.to_datetime(['2020-03-15', '2021-06-09', '2023-01-19',
                             '2023-11-04', '2024-08-29', '2025-02-13'])
})

In [ ]:
users.head()

,user_id,name,signup_date
0,1,Alice,2019-03-15
1,2,Bob,2020-06-10
2,3,Charlie,2021-01-20
3,4,Diana,2022-11-05
4,5,Eve,2023-08-30


In [ ]:
subscriptions.head()

,subscription_id,user_id,start_date,end_date
0,1,1,2019-03-16,2020-03-15
1,2,2,2020-06-10,2021-06-09
2,3,3,2021-01-20,2023-01-19
3,4,4,2022-11-05,2023-11-04
4,5,5,2023-08-30,2024-08-29


In [ ]:
years = pd.DataFrame({'year': pd.date_range(start='2019', end='2024', freq='YS').year})
df_new = years.merge(users.assign(year=users['signup_date'].dt.year).groupby('year', as_index=False).agg(count_new=('user_id', 'count')), how='left')
df_sub = subscriptions.assign(start_year=subscriptions['start_date'].dt.year, end_year=subscriptions['end_date'].dt.year).merge(years, how='cross')
df_sub = df_sub[(df_sub['year'] >= df_sub['start_year']) & (df_sub['year'] <= df_sub['end_year'])]
df_sub = df_sub.groupby('year', as_index=False).agg(count_active=('user_id', 'nunique'))
df_new.merge(df_sub, how='left').fillna(0).sort_values(by='year', ascending=False)


,year,count_new,count_active
5,2024,1,2
4,2023,1,3
3,2022,1,2
2,2021,1,2
1,2020,1,2
0,2019,1,1


In [ ]:
years_list = range(2019, 2025)
result = pd.DataFrame({'year': years_list})

# Считаем новых (группировка — это быстро)
new_users = users['signup_date'].dt.year.value_counts()
result['count_new'] = result['year'].map(new_users).fillna(0)

# 2. Считаем активных БЕЗ cross join
s = subscriptions.assign(
    start_y=subscriptions['start_date'].dt.year,
    end_y=subscriptions['end_date'].dt.year
)

def count_active_for_year(y):
    # Фильтруем оригинальный DataFrame только по условию для конкретного года
    mask = (s['start_y'] <= y) & (s['end_y'] >= y)
    return s.loc[mask, 'user_id'].nunique()

# Применяем функцию к каждой строке справочника (всего 6 итераций!)
result['count_active'] = result['year'].apply(count_active_for_year)

Есть таблица событий пользователей events с разными типами событий:
- install — установка приложения
- open_app — открытие приложения
- purchase — покупка   
Для каждого пользователя необходимо:
- Найти самое первое событие install.
-Найти самое первое событие purchase.
- Посчитать, сколько полных дней прошло между первым инсталлом и первой покупкой (целое число).
- Вывести сумму выручки (revenue) именно из этой первой покупки.
- Если у пользователя не было покупки, он не должен попадать в результирующую таблицу.   
Ожидаемые колонки в результирующем запросе:
- user_id
- days_diff — количество полных дней между установкой и покупкой
- first_revenue — выручка с первой покупки  
Отсортируйте результат по колонкам
user_id (по возрастанию)

In [ ]:
import pandas as pd
from datetime import datetime
data = [
    {'user_id': 'user_1', 'event_type': 'install', 'event_time': '2024-01-01 10:00:00', 'revenue': None},
    {'user_id': 'user_1', 'event_type': 'open_app', 'event_time': '2024-01-01 10:05:00', 'revenue': None},
    {'user_id': 'user_1', 'event_type': 'purchase', 'event_time': '2024-01-05 12:00:00', 'revenue': 9.99},
    {'user_id': 'user_1', 'event_type': 'purchase', 'event_time': '2024-01-10 14:30:00', 'revenue': 19.99},
    {'user_id': 'user_2', 'event_type': 'install', 'event_time': '2024-02-01 09:00:00', 'revenue': None},
    {'user_id': 'user_2', 'event_type': 'open_app', 'event_time': '2024-02-01 09:10:00', 'revenue': None},
    {'user_id': 'user_3', 'event_type': 'install', 'event_time': '2024-03-01 08:00:00', 'revenue': None},
    {'user_id': 'user_3', 'event_type': 'purchase', 'event_time': '2024-03-20 18:00:00', 'revenue': 4.99},
    {'user_id': 'user_3', 'event_type': 'purchase', 'event_time': '2024-04-01 11:00:00', 'revenue': 14.99},
    {'user_id': 'user_4', 'event_type': 'install', 'event_time': '2024-01-15 13:00:00', 'revenue': None},
    {'user_id': 'user_4', 'event_type': 'purchase', 'event_time': '2024-01-16 09:00:00', 'revenue': 2.99}
]
events = pd.DataFrame(data)
events['event_time'] = pd.to_datetime(events['event_time'])
events.head()

,user_id,event_type,event_time,revenue
0,user_1,install,2024-01-01 10:00:00,NaN
1,user_1,open_app,2024-01-01 10:05:00,NaN
2,user_1,purchase,2024-01-05 12:00:00,9.99
3,user_1,purchase,2024-01-10 14:30:00,19.99
4,user_2,install,2024-02-01 09:00:00,NaN


In [ ]:
installs = events[events['event_type'] == 'install'].groupby('user_id', as_index=False).agg(min_ins=('event_time', 'min'))
purchases = events[events['event_type'] == 'purchase'].rename(columns={'event_time': 'min_pur'})
purchases['rank_time_pur'] = purchases.groupby('user_id')['min_pur'].transform('rank')
result = purchases[purchases['rank_time_pur'] == 1].merge(installs)
result.assign(days_diff=(result['min_pur'] - result['min_ins']))[['user_id', 'days_diff', 'revenue']].sort_values('user_id')

,user_id,days_diff,revenue
0,user_1,4 days 02:00:00,9.99
1,user_3,19 days 10:00:00,4.99
2,user_4,0 days 20:00:00,2.99


In [ ]:
# 1. Считаем установки (сразу получаем Series, где индекс - user_id)
installs = events[events['event_type'] == 'install'].groupby('user_id')['event_time'].min()

# 2. Считаем покупки: находим индекс строки с первой покупкой для каждого юзера
# idxmin() найдет индекс первой строки с минимальным временем
first_pur_idx = events[events['event_type'] == 'purchase'].groupby('user_id')['event_time'].idxmin()
purchases = events.loc[first_pur_idx, ['user_id', 'event_time', 'revenue']].rename(columns={'event_time': 'min_pur'})

# 3. Объединяем и считаем разницу
# merge по индексу (user_id) в pandas работает очень быстро
result = purchases.merge(installs.rename('min_ins'), on='user_id')

# Считаем разницу и выводим нужные колонки
result['days_diff'] = (result['min_pur'] - result['min_ins']).dt.days
result[['user_id', 'days_diff', 'revenue']].sort_values('user_id')

,user_id,days_diff,revenue
0,user_1,4,9.99
1,user_3,19,4.99
2,user_4,0,2.99


 Есть таблица events, содержащая данные о событиях пользователей, включая покупки (event_type = 'purchase').
Для каждого пользователя необходимо проанализировать интервалы между покупками.  
Требования:
- Оставить только пользователей, у которых было 3 и более покупок.
- Для каждого такого пользователя найти максимальный разрыв (в днях) между двумя подряд идущими покупками.
- Если максимальный разрыв превышает 30 дней, считать, что пользователь ушёл (churned = true), иначе — остался (churned = false).  
Ожидаемые колонки в результирующем запросе:
- user_id
- max_gap_days
- churned (1 или 0, либо 'true'/'false')  
Отсортируйте результат по колонкам
user_id (по возрастанию)

In [ ]:
import pandas as pd
from datetime import datetime

data = [
    ['user_1', 'install', '2024-01-01 10:00:00', None],
    ['user_1', 'open_app', '2024-01-01 10:05:00', None],
    ['user_1', 'purchase', '2024-01-05 12:00:00', 9.99],
    ['user_1', 'purchase', '2024-01-10 14:30:00', 19.99],
    ['user_2', 'install', '2024-02-01 09:00:00', None],
    ['user_2', 'open_app', '2024-02-01 09:10:00', None],
    ['user_3', 'install', '2024-03-01 08:00:00', None],
    ['user_3', 'purchase', '2024-03-20 18:00:00', 4.99],
    ['user_3', 'purchase', '2024-04-01 11:00:00', 14.99],
    ['user_3', 'purchase', '2024-05-15 10:00:00', 25.0],
    ['user_4', 'install', '2024-01-15 13:00:00', None],
    ['user_4', 'purchase', '2024-01-16 09:00:00', 2.99],
    ['user_4', 'purchase', '2024-01-20 10:00:00', 5.0],
    ['user_4', 'purchase', '2024-02-25 10:00:00', 5.0]
]

events = pd.DataFrame(
    data,
    columns=['user_id', 'event_type', 'event_time', 'revenue']
)
events['event_time'] = pd.to_datetime(events['event_time'])
events.head()

,user_id,event_type,event_time,revenue
0,user_1,install,2024-01-01 10:00:00,NaN
1,user_1,open_app,2024-01-01 10:05:00,NaN
2,user_1,purchase,2024-01-05 12:00:00,9.99
3,user_1,purchase,2024-01-10 14:30:00,19.99
4,user_2,install,2024-02-01 09:00:00,NaN


In [ ]:
df = events[events['event_type'] == 'purchase'].sort_values(['user_id', 'event_time'])
df['pur_count'] = df.groupby('user_id')['event_time'].transform('count')
df = df[df['pur_count'] >= 3]
df['next_pur'] = df.groupby('user_id')['event_time'].shift(-1)
df['gap'] = (pd.to_datetime(df['next_pur']) - pd.to_datetime(df['event_time'])).dt.days
result = df.groupby('user_id', as_index=False).agg(
    max_gap_days=('gap', 'max')
)
result['churned'] = (result['max_gap_days'] > 30).astype(int)
result

,user_id,max_gap_days,churned
0,user_3,43.0,1
1,user_4,36.0,1


Есть данные о заказах, магазинах и товарах в заказах. Необходимо вывести список клиентов (client_id), которые:
- купили товар с item_id = '1'
- И купили товар с item_id = '3'
- НО при этом не покупали товар с item_id = '2'  
Пользователь считается купившим товар, если в любом из его заказов присутствует соответствующий item_id.  
Ожидаемые колонки в результирующем запросе:
- client_id  
Отсортируйте результат по колонке
client_id (по возрастанию)

In [ ]:
import pandas as pd
orders = pd.DataFrame({
    'created_date': [
        '2025-07-01 10:00:00',
        '2025-07-02 12:30:00',
        '2025-07-03 09:15:00',
        '2025-07-04 14:20:00',
        '2025-07-05 16:45:00'
    ],
    'store_id': ['store_1', 'store_2', 'store_1', 'store_3', 'store_2'],
    'order_id': ['order_1', 'order_2', 'order_3', 'order_4', 'order_5'],
    'client_id': ['client_1', 'client_1', 'client_2', 'client_3', 'client_4'],
    'order_sum': [300.0, 150.0, 200.0, 400.0, 250.0]
})


In [ ]:
items = pd.DataFrame({
    'order_id': ['order_1', 'order_1', 'order_2', 'order_3',
                'order_4', 'order_4', 'order_4', 'order_5', 'order_5'],
    'item_id': [1, 3, 1, 2, 1, 2, 3, 1, 3],
    'item_cost': [100.0, 200.0, 150.0, 200.0,
                 150.0, 100.0, 150.0, 100.0, 150.0]
})

In [ ]:
orders.head()

,created_date,store_id,order_id,client_id,order_sum
0,2025-07-01 10:00:00,store_1,order_1,client_1,300.0
1,2025-07-02 12:30:00,store_2,order_2,client_1,150.0
2,2025-07-03 09:15:00,store_1,order_3,client_2,200.0
3,2025-07-04 14:20:00,store_3,order_4,client_3,400.0
4,2025-07-05 16:45:00,store_2,order_5,client_4,250.0


In [ ]:
items.head()

,order_id,item_id,item_cost
0,order_1,1,100.0
1,order_1,3,200.0
2,order_2,1,150.0
3,order_3,2,200.0
4,order_4,1,150.0


In [ ]:
df = orders.merge(items)
df = df.groupby('client_id', as_index=False).agg(
    sum_i1=('item_id', lambda x: (x == 1).sum()),
    sum_i3=('item_id', lambda x:(x == 3).sum()),
    sum_i2=(('item_id', lambda x:(x == 2).sum()))
)
df[(df['sum_i1'] > 0) & (df['sum_i3'] > 0) & (df['sum_i2'] == 0)][['client_id']]

,client_id
0,client_1
3,client_4


In [ ]:
# 1. Оставляем только нужные item_id сразу после мерджа
ids = [1, 2, 3]
df = orders.merge(items.query('item_id in @ids'))

# 2. Делаем сводную таблицу (это быстрее, чем agg с lambda)
pivot = df.pivot_table(
    index='client_id',
    columns='item_id',
    aggfunc='size',
    fill_value=0
)

# 3. Фильтруем результат
# Убеждаемся, что все колонки (1, 2, 3) присутствуют, даже если их не было в данных
for i in ids:
    if i not in pivot.columns: pivot[i] = 0

result = pivot[(pivot[1] > 0) & (pivot[3] > 0) & (pivot[2] == 0)].reset_index()[['client_id']]

Для каждого клиента выведите client_id и order_id самого первого заказа этого клиента. Самым первым считается заказ с минимальной датой created_date.
Если у клиента несколько заказов в одну и ту же минимальную дату, выберите заказ с наименьшим order_id.  
Ожидаемые колонки в результирующем запросе:
- client_id
- order_id  
Отсортируйте результат по колонке client_id (по возрастанию)

In [1]:
import pandas as pd
data = {
    'created_date': [
        '2025-07-01 10:00:00',
        '2025-07-02 12:30:00',
        '2025-07-03 09:15:00',
        '2025-07-04 14:20:00',
        '2025-07-05 16:45:00'
    ],
    'store_id': ['store_1', 'store_2', 'store_1', 'store_3', 'store_2'],
    'order_id': ['order_1', 'order_2', 'order_3', 'order_4', 'order_5'],
    'client_id': ['client_1', 'client_1', 'client_2', 'client_3', 'client_4'],
    'order_sum': [300.0, 150.0, 200.0, 400.0, 250.0]
}
orders = pd.DataFrame(data)
orders['created_date'] = pd.to_datetime(orders['created_date'])
orders.head()

,created_date,store_id,order_id,client_id,order_sum
0,2025-07-01 10:00:00,store_1,order_1,client_1,300.0
1,2025-07-02 12:30:00,store_2,order_2,client_1,150.0
2,2025-07-03 09:15:00,store_1,order_3,client_2,200.0
3,2025-07-04 14:20:00,store_3,order_4,client_3,400.0
4,2025-07-05 16:45:00,store_2,order_5,client_4,250.0


In [5]:
df = orders.sort_values(['client_id', 'created_date', 'order_id'])
df.groupby('client_id', as_index=False).head(1)[['client_id', 'order_id']]

,client_id,order_id
0,client_1,order_1
2,client_2,order_3
3,client_3,order_4
4,client_4,order_5


In [7]:
orders.sort_values(['client_id', 'created_date', 'order_id']).drop_duplicates(subset=['client_id'], keep='first')[['client_id', 'order_id']]

,client_id,order_id
0,client_1,order_1
2,client_2,order_3
3,client_3,order_4
4,client_4,order_5


 Для всех клиентов, которые совершили свой первый заказ в городе Moscow до 1 июля 2024 года (то есть created_date строго меньше '2024-07-01 00:00:00'), выведите client_id и order_id самого дорогого заказа этого клиента. Самый дорогой заказ — это заказ с максимальным order_sum. Если таких заказов несколько, выберите заказ:
- с наибольшей датой created_date
- если и даты совпали — с наибольшим order_id  
Ожидаемые колонки в результирующем запросе: 
- client_id
- order_id   
Отсортируйте результат по колонке client_id (по возрастанию)

In [3]:
import pandas as pd
orders = pd.DataFrame({
    'created_date': [
        '2023-06-01 10:00:00',
        '2023-06-15 12:30:00',
        '2023-07-03 09:15:00',
        '2023-05-20 14:20:00',
        '2024-01-05 16:45:00',
        '2024-08-01 10:00:00'
    ],
    'store_id': ['store_1', 'store_2', 'store_1', 'store_1', 'store_2', 'store_1'],
    'order_id': ['order_1', 'order_2', 'order_3', 'order_4', 'order_5', 'order_6'],
    'client_id': ['client_1', 'client_1', 'client_1', 'client_2', 'client_2', 'client_3'],
    'order_sum': [300.0, 150.0, 500.0, 400.0, 250.0, 600.0]
})
orders.head()

,created_date,store_id,order_id,client_id,order_sum
0,2023-06-01 10:00:00,store_1,order_1,client_1,300.0
1,2023-06-15 12:30:00,store_2,order_2,client_1,150.0
2,2023-07-03 09:15:00,store_1,order_3,client_1,500.0
3,2023-05-20 14:20:00,store_1,order_4,client_2,400.0
4,2024-01-05 16:45:00,store_2,order_5,client_2,250.0


In [4]:
stores = pd.DataFrame({
    'store_id': ['store_1', 'store_2', 'store_3'],
    'city': ['Moscow', 'Saint Petersburg', 'Kazan']
})
stores.head()

,store_id,city
0,store_1,Moscow
1,store_2,Saint Petersburg
2,store_3,Kazan


In [10]:
target_ids = orders[orders['created_date'] <  '2024-07-01 00:00:00'].merge(stores[stores['city'] == 'Moscow'])['client_id'].unique()
orders[orders['client_id'].isin(target_ids)].sort_values(['client_id', 'order_sum', 'created_date', 'order_id'], ascending=[True, False, False, False])\
    .drop_duplicates(subset=['client_id'], keep='first')[['client_id', 'order_id']]

,client_id,order_id
2,client_1,order_3
3,client_2,order_4


In [11]:
moscow_stores = stores.loc[stores['city'] == 'Moscow', 'store_id']
target_ids = orders[
    (orders['created_date'] < '2024-07-01') & 
    (orders['store_id'].isin(moscow_stores))
]['client_id'].unique()
orders[orders['client_id'].isin(target_ids)].sort_values(['client_id', 'order_sum', 'created_date', 'order_id'], ascending=[True, False, False, False])\
    .drop_duplicates(subset=['client_id'], keep='first')[['client_id', 'order_id']]

,client_id,order_id
2,client_1,order_3
3,client_2,order_4
